# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 2.5 Run Level Settings

Change only `RUN_LEVEL` to switch between quick Colab checks (`LIGHT`) and longer presentation runs (`BASIC`).


In [ ]:
# Colab run-size settings: LIGHT is quick, BASIC is better for presentation logs.
RUN_LEVEL = "BASIC"  # "LIGHT" or "BASIC"

RUN_CONFIGS = {
    "LIGHT": {
        "CORPUS_LIMIT": 500_000,
        "VAL_CORPUS_LIMIT": 50_000,
        "VOCAB_SIZE": 2000,
        "CONTEXT_LENGTH": 64,
        "EMB_DIM": 128,
        "N_HEADS": 4,
        "N_LAYERS": 2,
        "BATCH_SIZE": 8,
        "NUM_EPOCHS": 2,
        "EVAL_FREQ": 100,
        "EVAL_ITER": 10,
        "CKPT_FREQ": 500,
        "SENTIMENT_TRAIN_LIMIT": 3_000,
        "SENTIMENT_VAL_LIMIT": 1_000,
        "SENTIMENT_TEST_LIMIT": 1_000,
        "FINETUNE_EPOCHS": 2,
    },
    "BASIC": {
        "CORPUS_LIMIT": 1_500_000,
        "VAL_CORPUS_LIMIT": 150_000,
        "VOCAB_SIZE": 3000,
        "CONTEXT_LENGTH": 128,
        "EMB_DIM": 192,
        "N_HEADS": 4,
        "N_LAYERS": 4,
        "BATCH_SIZE": 8,
        "NUM_EPOCHS": 2,
        "EVAL_FREQ": 100,
        "EVAL_ITER": 10,
        "CKPT_FREQ": 500,
        "SENTIMENT_TRAIN_LIMIT": 10_000,
        "SENTIMENT_VAL_LIMIT": 2_000,
        "SENTIMENT_TEST_LIMIT": 2_000,
        "FINETUNE_EPOCHS": 2,
    },
}

if RUN_LEVEL not in RUN_CONFIGS:
    raise ValueError(f"RUN_LEVEL must be one of {list(RUN_CONFIGS)}: {RUN_LEVEL}")

_cfg = RUN_CONFIGS[RUN_LEVEL]
CORPUS_LIMIT = min(_cfg["CORPUS_LIMIT"], len(corpus)) if corpus else _cfg["CORPUS_LIMIT"]
VAL_CORPUS_LIMIT = min(_cfg["VAL_CORPUS_LIMIT"], len(val_corpus)) if val_corpus else _cfg["VAL_CORPUS_LIMIT"]
VOCAB_SIZE = _cfg["VOCAB_SIZE"]
CONTEXT_LENGTH = _cfg["CONTEXT_LENGTH"]
EMB_DIM = _cfg["EMB_DIM"]
N_HEADS = _cfg["N_HEADS"]
N_LAYERS = _cfg["N_LAYERS"]
BATCH_SIZE = _cfg["BATCH_SIZE"]
NUM_EPOCHS = _cfg["NUM_EPOCHS"]
EVAL_FREQ = _cfg["EVAL_FREQ"]
EVAL_ITER = _cfg["EVAL_ITER"]
CKPT_FREQ = _cfg["CKPT_FREQ"]
SENTIMENT_TRAIN_LIMIT = _cfg["SENTIMENT_TRAIN_LIMIT"]
SENTIMENT_VAL_LIMIT = _cfg["SENTIMENT_VAL_LIMIT"]
SENTIMENT_TEST_LIMIT = _cfg["SENTIMENT_TEST_LIMIT"]
FINETUNE_EPOCHS = _cfg["FINETUNE_EPOCHS"]

PRETRAIN_LR = 3e-4
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 0.1


def make_gpt_config(drop_rate: float = 0.1) -> dict:
    return {
        "vocab_size": VOCAB_SIZE,
        "context_length": CONTEXT_LENGTH,
        "emb_dim": EMB_DIM,
        "n_heads": N_HEADS,
        "n_layers": N_LAYERS,
        "drop_rate": drop_rate,
        "qkv_bias": False,
    }


def get_or_train_tokenizer():
    from bpe import BPETokenizer

    if not corpus:
        raise ValueError("corpus is empty. Run the data preparation cells first.")

    vocab_path = repo_dir / "data" / f"vocab_bpe_{VOCAB_SIZE}.json"
    tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)
    if vocab_path.exists():
        tokenizer.load(vocab_path)
        print("BPE vocab loaded:", vocab_path)
    else:
        print(f"BPE vocab training: first {CORPUS_LIMIT:,} chars, vocab_size={VOCAB_SIZE}")
        tokenizer.train(corpus[:CORPUS_LIMIT])
        tokenizer.save(vocab_path)
        print("BPE vocab saved:", vocab_path)
    return tokenizer


print("RUN_LEVEL:", RUN_LEVEL)
print("pretrain:", {"chars": CORPUS_LIMIT, "vocab": VOCAB_SIZE, "context": CONTEXT_LENGTH, "batch": BATCH_SIZE, "epochs": NUM_EPOCHS})
print("model:", make_gpt_config())
print("sentiment limits:", SENTIMENT_TRAIN_LIMIT, SENTIMENT_VAL_LIMIT, SENTIMENT_TEST_LIMIT)


## 2.6 Colab Actual Run Checklist

Use this notebook in two passes.

1. Quick check: set `RUN_LEVEL = "LIGHT"`, run the cells from top to bottom, and confirm the smoke checks pass.
2. Presentation run: set `RUN_LEVEL = "BASIC"`, then turn on the long training cells below.
3. Pretraining: in `## 7.5 Actual Pretraining Run`, set `RUN_ACTUAL_PRETRAIN = True`.
4. Fine-tuning: in `## 8.5 Actual Sentiment Fine-Tuning Run`, set `RUN_ACTUAL_FINETUNE = True`.
5. Logs to check:
   - `logs/pretrain_metrics.jsonl`: train loss and validation loss
   - `logs/pretrain_samples.jsonl`: generated samples
   - `logs/sentiment_metrics.jsonl`: train/validation/test loss and accuracy

Useful Colab checks:

```python
!tail -n 20 logs/pretrain_metrics.jsonl
!tail -n 5 logs/pretrain_samples.jsonl
!tail -n 20 logs/sentiment_metrics.jsonl
```


## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# Check BPE encode/decode with the current RUN_LEVEL tokenizer.
try:
    tokenizer = get_or_train_tokenizer()
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except (NotImplementedError, ValueError) as e:
    print("BPE is not ready or data is missing:", e)


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = get_or_train_tokenizer()
    token_ids = tokenizer.encode(corpus[:CORPUS_LIMIT])
    loader = create_dataloader(
        token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=True,
    )
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(
        vocab_size=VOCAB_SIZE,
        emb_dim=EMB_DIM,
        context_length=CONTEXT_LENGTH,
        drop_rate=0.1,
    )
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except (NotImplementedError, ValueError) as e:
    print("Dataset/Embedding is not ready or data is missing:", e)


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = make_gpt_config(drop_rate=0.1)
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, min(16, config["context_length"])))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model is not ready:", e)


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

### Optional - Colab Checkpoints

Colab 런타임이 끊겨도 이어서 학습할 수 있도록 Google Drive에 체크포인트를 저장합니다.


In [ ]:
from pathlib import Path
import sys

CKPT_DIR = repo_dir / "checkpoints"
if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    CKPT_DIR = Path("/content/drive/MyDrive/minigpt_checkpoints") / repo_dir.name

CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint dir:", CKPT_DIR)


def get_latest_checkpoint(ckpt_dir: Path):
    candidates = sorted(ckpt_dir.glob("*.pt"))
    return candidates[-1] if candidates else None


latest_ckpt = get_latest_checkpoint(CKPT_DIR)
print("Latest checkpoint:", latest_ckpt if latest_ckpt is not None else "None")


In [ ]:
# Run one loss/backward smoke test with the current RUN_LEVEL settings.
try:
    import torch
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = get_or_train_tokenizer()
    token_ids = tokenizer.encode(corpus[:CORPUS_LIMIT])
    loader = create_dataloader(
        token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=True,
    )
    inp, tgt = next(iter(loader))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GPTModel(make_gpt_config(drop_rate=0.1)).to(device)
    loss = calc_loss_batch(inp, tgt, model, device)
    loss.backward()
    print("device:", device)
    print("smoke loss:", loss.item())
except (NotImplementedError, ValueError) as e:
    print("Pretraining utilities are not ready or data is missing:", e)


## 7.5 Actual Pretraining Run

Set `RUN_ACTUAL_PRETRAIN = True` to train with the selected `RUN_LEVEL` and write `logs/pretrain_metrics.jsonl` plus `logs/pretrain_samples.jsonl`.


In [ ]:
RUN_ACTUAL_PRETRAIN = False  # Change to True when you want the longer run.

if RUN_ACTUAL_PRETRAIN:
    import torch
    from dataset import create_dataloader
    from model import GPTModel
    from train import load_checkpoint, train_model

    tokenizer = get_or_train_tokenizer()
    train_token_ids = tokenizer.encode(corpus[:CORPUS_LIMIT])
    val_text = val_corpus[:VAL_CORPUS_LIMIT] if val_corpus else corpus[:max(10_000, CORPUS_LIMIT // 10)]
    val_token_ids = tokenizer.encode(val_text)

    train_loader = create_dataloader(
        train_token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=True,
        drop_last=True,
    )
    val_loader = create_dataloader(
        val_token_ids,
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE,
        stride=CONTEXT_LENGTH,
        shuffle=False,
        drop_last=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GPTModel(make_gpt_config(drop_rate=0.1)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
    start_epoch = 0
    global_step = 0

    latest_ckpt = get_latest_checkpoint(CKPT_DIR)
    if latest_ckpt is not None:
        start_epoch, global_step = load_checkpoint(model, optimizer, str(latest_ckpt), device)
        print(f"continue from epoch={start_epoch}, global_step={global_step}")

    train_losses = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        device,
        num_epochs=NUM_EPOCHS,
        eval_freq=EVAL_FREQ,
        eval_iter=EVAL_ITER,
        start_context="이 영화는",
        tokenizer=tokenizer,
        ckpt_freq=CKPT_FREQ,
        start_epoch=start_epoch,
        global_step=global_step,
        ckpt_dir=CKPT_DIR,
        metrics_path=repo_dir / "logs" / "pretrain_metrics.jsonl",
        sample_path=repo_dir / "logs" / "pretrain_samples.jsonl",
    )

    print("pretrain losses:", train_losses)
    print("metrics:", repo_dir / "logs" / "pretrain_metrics.jsonl")
    print("samples:", repo_dir / "logs" / "pretrain_samples.jsonl")
else:
    print("Set RUN_ACTUAL_PRETRAIN = True and rerun this cell to start actual pretraining.")


In [ ]:
# Print 10 generated samples after pretraining.
if "model" in globals() and "tokenizer" in globals():
    from train import generate_and_print_sample

    for sample_idx in range(10):
        print(f"\n=== Sample {sample_idx + 1} ===")
        generate_and_print_sample(
            model,
            tokenizer,
            device,
            start_context="이 영화는",
            max_new_tokens=30,
            context_size=model.config["context_length"],
        )
else:
    print("Run the pretraining cell first so model/tokenizer are available.")


## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 8.5 Actual Sentiment Fine-Tuning Run

Set `RUN_ACTUAL_FINETUNE = True` to log train/validation/test loss and accuracy to `logs/sentiment_metrics.jsonl`.


In [ ]:
RUN_ACTUAL_FINETUNE = False  # Change to True when you want actual fine-tuning.

if RUN_ACTUAL_FINETUNE:
    import json
    import torch
    from torch.utils.data import DataLoader
    from finetune import (
        GPTForSequenceClassification,
        ReviewSentimentDataset,
        evaluate_sentiment,
        train_epoch_sentiment,
    )
    from model import GPTModel

    def read_jsonl(path, limit=None):
        rows = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                rows.append(json.loads(line))
                if limit is not None and len(rows) >= limit:
                    break
        return rows

    tokenizer = globals().get("tokenizer") or get_or_train_tokenizer()
    if "model" not in globals():
        print("No pretrained model variable was found, so a new GPTModel is created.")
        model = GPTModel(make_gpt_config(drop_rate=0.1))

    train_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_train.jsonl", SENTIMENT_TRAIN_LIMIT)
    val_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_val.jsonl", SENTIMENT_VAL_LIMIT)
    test_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_test.jsonl", SENTIMENT_TEST_LIMIT)

    train_ds = ReviewSentimentDataset(train_data, tokenizer, max_length=CONTEXT_LENGTH)
    val_ds = ReviewSentimentDataset(val_data, tokenizer, max_length=CONTEXT_LENGTH)
    test_ds = ReviewSentimentDataset(test_data, tokenizer, max_length=CONTEXT_LENGTH)

    train_cls_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_cls_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_cls_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clf_model = GPTForSequenceClassification(model, num_labels=2).to(device)
    clf_optimizer = torch.optim.AdamW(clf_model.parameters(), lr=FINETUNE_LR)

    for epoch in range(FINETUNE_EPOCHS):
        train_loss, train_acc = train_epoch_sentiment(
            clf_model,
            train_cls_loader,
            clf_optimizer,
            device,
            epoch=epoch,
            metrics_path=repo_dir / "logs" / "sentiment_metrics.jsonl",
        )
        val_loss, val_acc = evaluate_sentiment(
            clf_model,
            val_cls_loader,
            device,
            split="val",
            epoch=epoch,
            metrics_path=repo_dir / "logs" / "sentiment_metrics.jsonl",
        )
        print(f"epoch {epoch}: train loss={train_loss:.4f}, train acc={train_acc:.4f}, val loss={val_loss:.4f}, val acc={val_acc:.4f}")

    test_loss, test_acc = evaluate_sentiment(
        clf_model,
        test_cls_loader,
        device,
        split="test",
        metrics_path=repo_dir / "logs" / "sentiment_metrics.jsonl",
    )
    print(f"test loss={test_loss:.4f}, test acc={test_acc:.4f}")
    print("metrics:", repo_dir / "logs" / "sentiment_metrics.jsonl")
else:
    print("Set RUN_ACTUAL_FINETUNE = True and rerun this cell to start actual fine-tuning.")


## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")